<a href="https://colab.research.google.com/github/aminopropylamine/Monte-Carlo-Portfolio-Optimisation/blob/main/MCPO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Step 1: Environment Setup

In [ ]:
!pip install yfinance --upgrade

import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
print("Success: Libraries imported and environment is ready.")

Success: Libraries imported and environment is ready.


#Step 2: Data Acquisition
##In this step, we define our portfolio assets and download 10 years of historical data.

In [ ]:
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'JPM', 'V', 'KO', 'PEP', 'XOM', 'JNJ', 'PFE', 'WMT', 'PG', 'NKE']
start_date = '2016-05-01'
end_date = '2026-05-01'
raw_data = yf.download(tickers, start=start_date, end=end_date)

if 'Adj Close' in raw_data.columns:
    data = raw_data['Adj Close']
else:
    print("\nWarning: 'Adj Close' not found, switching to 'Close' prices.")
    data = raw_data['Close']

print("Success: Historical data downloaded.")
data.head()

[*********************100%***********************]  15 of 15 completed



Success: Historical data downloaded.


Ticker,AAPL,AMZN,GOOGL,JNJ,JPM,KO,MSFT,NKE,PEP,PFE,PG,TSLA,V,WMT,XOM
Date,,,,,,,,,,,,,,,
2016-05-02,21.207630,34.192501,35.427689,85.639664,48.889774,32.878975,44.531193,52.238087,76.378181,20.085957,61.641232,16.120001,73.144295,18.878222,57.653793
2016-05-03,21.556414,33.566002,35.131634,85.594078,47.947075,32.776649,43.800869,52.176731,76.474174,20.637094,61.740181,15.488000,71.960342,18.713425,56.993996
2016-05-04,21.332195,33.544998,35.276936,85.237106,47.188324,32.878975,43.880070,51.817318,76.813866,20.453373,62.120850,14.837333,71.848495,18.766499,56.884033
2016-05-05,21.245609,32.954498,35.442574,85.738396,46.935417,32.937469,43.941650,51.010815,76.954178,20.557484,61.892433,14.102000,72.118851,18.772093,56.948734
2016-05-06,21.127121,33.697498,35.961773,85.632057,47.211319,33.127510,44.337616,51.221195,77.508018,20.563610,62.524300,14.328667,72.454430,19.062561,57.252739


#Step 3: Data Cleaning and Preprocessing
##Handling missing values and ensuring data integrity.

In [ ]:
print("Missing values count per asset:")
print(data.isnull().sum())

data = data.fillna(method='ffill').dropna()
print(f"Success: Data cleaned. Final dataset shape: {data.shape}")

Missing values count per asset:
Ticker
AAPL     0
AMZN     0
GOOGL    0
JNJ      0
JPM      0
KO       0
MSFT     0
NKE      0
PEP      0
PFE      0
PG       0
TSLA     0
V        0
WMT      0
XOM      0
dtype: int64
Success: Data cleaned. Final dataset shape: (2514, 15)


/tmp/ipykernel_13431/1565746198.py:6: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  data = data.fillna(method='ffill').dropna()


#Step 4: Financial Metrics Calculation
##Calculating daily returns, annualized mean returns, and the covariance matrix.

In [ ]:
daily_returns = data.pct_change().dropna()
annual_mean_returns = daily_returns.mean() * 252
cov_matrix = daily_returns.cov() * 252

print("Success: Returns and Covariance matrix calculated.")
print("\nAnnualized Mean Returns (dtype: float64):")
print(annual_mean_returns)

Success: Returns and Covariance matrix calculated.

Annualized Mean Returns (dtype: float64):
Ticker
AAPL     0.297358
AMZN     0.257924
GOOGL    0.281368
JNJ      0.115955
JPM      0.223676
KO       0.104149
MSFT     0.258396
NKE      0.035644
PEP      0.092409
PFE      0.056967
PG       0.105087
TSLA     0.491128
V        0.180931
WMT      0.218693
XOM      0.138051
dtype: float64


#Step 5: Portfolio Performance Logic
##Defining a function to calculate the return and volatility for a given set of weights.

In [ ]:
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.sum(mean_returns * weights)
    std_dev = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std_dev

print("Success: Performance function is ready for Monte Carlo simulation.")

Success: Performance function is ready for Monte Carlo simulation.
